In [1]:
import netket as nk
import numpy as np
import matplotlib.pyplot as plt
import json
from pyscf import gto, scf, fci
import netket.experimental as nkx

# 设置H2分子的几何构型
bond_length = 1.74  # H2平衡键长（埃）
geometry = [
    ('H', (0., 0., 0.)),
    ('H', (bond_length, 0., 0.)),
]

# 创建分子对象，使用STO-3G基组
mol = gto.M(atom=geometry, basis='STO-3G')

# 进行Hartree-Fock计算
mf = scf.RHF(mol).run(verbose=0)
E_hf = mf.e_tot
print(f"Hartree-Fock能量: {E_hf:.8f} Ha")

# 进行FCI计算作为参考
cisolver = fci.FCI(mf)
E_fci, fcivec = cisolver.kernel()
print(f"FCI能量: {E_fci:.8f} Ha")

# 使用NetKet创建哈密顿量
ha = nkx.operator.from_pyscf_molecule(mol)

# 获取Hartree-Fock分子轨道系数
mo_coeff = mf.mo_coeff
print(f"分子轨道系数形状: {mo_coeff.shape}")

/opt/miniconda3/envs/Neural/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Hartree-Fock能量: -0.84390759 Ha
FCI能量: -0.96730577 Ha
分子轨道系数形状: (2, 2)


SpinOrbitalFermions 是 NetKet（特别是 netket ≥ v3.10 之后，或通过 netket.experimental）中用于描述具有自旋的费米子系统的希尔伯特空间（Hilbert space）类。它专为处理电子结构问题（如分子、固体中的多电子系统）而设计，天然支持泡利不相容原理和固定粒子数约束。

SpinOrbitalFermions 定义了一个由自旋轨道占据数构成的离散希尔伯特空间，每个自旋轨道只能被占据（1）或空（0），且满足：

费米子统计（自动通过占据数表示处理）
可选：固定总电子数、固定自旋向上/向下电子数


In [2]:
hi = nk.hilbert.SpinOrbitalFermions(
    n_orbitals=2,  # 总空间轨道数
    s = 1/2,
    n_fermions_per_spin=(1, 1)  # 每种自旋的电子数
)

# 创建采样器 - 使用费米子跳跃采样器
# 对于分子系统，我们使用完整的轨道图（完全连接）
cluster = [(0,1),(2,3)]
#

# g = nk.graph.Graph(edges=[(0,2),(1,3),(2,0),(3,1)])
g = nk.graph.Graph(edges=[(0,1),(2,3)])
# g = nk.graph.Graph(edges=cluster)
sa = nk.sampler.MetropolisFermionHop(
    hi, graph=g, n_chains=16, spin_symmetric=True, sweep_size=64
)

In [3]:
hi.all_states()

Array([[0, 1, 0, 1],
       [0, 1, 1, 0],
       [1, 0, 0, 1],
       [1, 0, 1, 0]], dtype=int8)

In [4]:
import jax
import jax.numpy as jnp
from flax import nnx

class FFN(nnx.Module):

    def __init__(self, N: int, alpha: int = 1, *, rngs: nnx.Rngs):
        """
        Construct a Feed-Forward Neural Network with a single hidden layer.
        This will serve as the Jastrow factor.

        Args:
            N: The number of input nodes (number of spins in the chain).
            alpha: The density of the hidden layer. The hidden layer will have
                N*alpha nodes.
            rngs: The random number generator seed.
        """
        self.alpha = alpha

        # We define a linear (or dense) layer with `alpha` times the number of input nodes
        # as output nodes.
        # We must pass forward the rngs object to the dense layer.
        self.linear = nnx.Linear(in_features=N, out_features=alpha * N, rngs=rngs)

    def __call__(self, x: jax.Array):

        # we apply the linear layer to the input
        y = self.linear(x)

        # the non-linearity is a simple ReLu
        y = nnx.relu(y)

        # sum the output
        return jnp.sum(y, axis=-1)


class FFNNJastrow(nnx.Module):
    """
    FFNN-Jastrow 模型：
    ψ(x) = det(x) * exp(FFNN(x))
    
    其中：
    - det(x) 是 Slater 行列式，提供反对称性
    - FFNN(x) 是前馈神经网络，作为 Jastrow 因子捕捉电子相关性
    """
    
    def __init__(self, N: int, alpha: int = 1, mo_coeff: jnp.ndarray = None, *, rngs: nnx.Rngs):
        """
        Args:
            N: The number of input nodes (number of spin-orbitals).
            alpha: The density of the hidden layer in FFNN.
            mo_coeff: Molecular orbital coefficients from Hartree-Fock.
            rngs: The random number generator seed.
        """
        self.N = N
        self.alpha = alpha
        self.mo_coeff = mo_coeff if mo_coeff is not None else jnp.eye(N)
        
        # FFNN 作为 Jastrow 因子
        self.ffnn = FFN(N=N, alpha=alpha, rngs=rngs)
    
    def compute_slater_determinant(self, x: jax.Array) -> jax.Array:
        """
        计算 Slater 行列式。
        
        Args:
            x: 占据数配置，形状为 (..., N)
        
        Returns:
            Slater 行列式的对数值
        """
        # 获取占据的轨道索引
        occupied = jnp.where(x == 1, size=self.N, fill_value=-1)[0]
        occupied = occupied[occupied >= 0]  # 过滤掉填充值
        
        # 如果没有占据的轨道，返回 0
        if len(occupied) == 0:
            return jnp.zeros(x.shape[:-1])
        
        # 从分子轨道系数中提取占据轨道的系数
        # 对于 H2，我们有 2 个空间轨道，每个轨道有自旋向上和向下
        # 所以总共有 4 个自旋轨道
        
        # 构建自旋轨道系数矩阵
        # 前两个是自旋向上，后两个是自旋向下
        n_spin_orbitals = self.N
        n_spatial_orbitals = n_spin_orbitals // 2
        
        # 构建自旋轨道系数
        spin_mo_coeff = jnp.zeros((n_spin_orbitals, n_spin_orbitals))
        for i in range(n_spatial_orbitals):
            # 自旋向上
            spin_mo_coeff = spin_mo_coeff.at[2*i, :n_spatial_orbitals].set(self.mo_coeff[:, i])
            # 自旋向下
            spin_mo_coeff = spin_mo_coeff.at[2*i+1, :n_spatial_orbitals].set(self.mo_coeff[:, i])
        
        # 提取占据轨道的系数
        det_matrix = spin_mo_coeff[occupied, :]
        
        # 计算行列式
        det = jnp.linalg.det(det_matrix)
        
        # 返回对数值，避免数值问题
        return jnp.log(jnp.abs(det) + 1e-10)
    
    def __call__(self, x: jax.Array) -> jax.Array:
        """
        计算 FFNN-Jastrow 波函数的对数值。
        
        ψ(x) = det(x) * exp(FFNN(x))
        log ψ(x) = log det(x) + FFNN(x)
        
        Args:
            x: 占据数配置，形状为 (..., N)
        
        Returns:
            波函数的对数值
        """
        # 计算 Slater 行列式的对数值
        log_det = self.compute_slater_determinant(x)
        
        # 计算 FFNN (Jastrow 因子)
        jastrow = self.ffnn(x)
        
        # 返回对数波函数
        return log_det + jastrow


# model = FFNNJastrow(N=N, alpha=1, mo_coeff=mo_coeff, rngs=nnx.Rngs(2))

# vstate = nk.vqs.MCState(sampler, model, n_samples=1008)

In [5]:
N = 4
ffnn_jastrow_model = FFNNJastrow(N=N, alpha=1, mo_coeff=mo_coeff, rngs=nnx.Rngs(2))
vs = nk.vqs.MCState(sa, ffnn_jastrow_model, n_discard_per_chain=10, n_samples=512)

# 设置优化器
opt = nk.optimizer.Sgd(learning_rate=0.05)
sr = nk.optimizer.SR(diag_shift=0.01)

# 创建VMC驱动器
gs = nk.driver.VMC(ha, opt, variational_state=vs, preconditioner=sr)

# 运行优化
exp_name = "h2_molecule_ffnn_jastrow"

NonConcreteBooleanIndexError: Array boolean indices must be concrete; got bool[4]

See https://docs.jax.dev/en/latest/errors.html#jax.errors.NonConcreteBooleanIndexError

In [ ]:
gs.run(300, out=exp_name)

In [ ]:
############## 绘图 #################
# 获取精确对角化能量（FCI能量）
ed_energies = np.array([E_fci])  # H2只有一个基态能量

# 读取日志数据
with open(f"{exp_name}.log") as f:
    data = json.load(f)

x = data["Energy"]["iters"]
y = data["Energy"]["Mean"]

# 绘制能量收敛曲线
plt.figure(figsize=(10, 6))
plt.axhline(ed_energies[0], color="red", linestyle="--", label=f"FCI energy = {E_fci:.6f} Ha")
plt.axhline(E_hf, color="green", linestyle="--", label=f"Hartree Fock energy = {E_hf:.6f} Ha")
plt.plot(x, y, 'b-', label="VMC energy (FFNN-Jastrow)")
plt.xlabel("Iterations")
plt.ylabel("Energy (Ha)")
plt.title("H2 molecule energy convergence (FFNN-Jastrow)")
plt.legend()
plt.grid(True)
plt.show()

# 打印最终结果
print(f"最终VMC能量 (FFNN-Jastrow): {y[-1]:.8f} Ha")
print(f"最终FCI能量: {E_fci:.8f} Ha")
print(f"最终Hartree-Fock能量: {E_hf:.8f} Ha")
print(f"与FCI能量误差: {abs(y[-1] - E_fci):.8f} Ha")
print(f"相对于Hartree-Fock的改进: {abs(y[-1] - E_hf):.8f} Ha")

In [ ]:
# 分析组态及其概率分布
import numpy as np
import matplotlib.pyplot as plt

# 1. 生成大量样本以获得准确的概率分布
n_samples = 10000  # 增加样本数量以获得更准确的概率
samples = vs.sample(n_samples=n_samples)

# 2. 将样本重塑为二维数组 (n_samples, n_orbitals)
samples_flat = samples.reshape(-1, samples.shape[-1])

# 3. 统计每个组态的出现次数
unique_configs, counts = np.unique(samples_flat, axis=0, return_counts=True)

# 4. 计算概率
probabilities = counts / np.sum(counts)

# 5. 按概率排序
sorted_indices = np.argsort(probabilities)[::-1]
sorted_configs = unique_configs[sorted_indices]
sorted_probs = probabilities[sorted_indices]

print("组态及其概率分布 (FFNN-Jastrow):")
print("组态(轨道占据)    概率      物理解释")
print("-" * 50)
for i, (config, prob) in enumerate(zip(sorted_configs[:10], sorted_probs[:10])):  # 显示前10个主要组态
    # 解释组态的物理意义
    occupied_orbitals = np.where(config == 1)[0]
    print(f"{config}         {prob:.6f}   电子占据轨道: {occupied_orbitals}")

In [ ]:
# 对比 FFNN 和 FFNN-Jastrow 的性能
import matplotlib.pyplot as plt
import json
import os

# 尝试读取 FFNN 的结果
ffnn_log_file = "h2_molecule_ffnn_layer2.log"
ffnn_jastrow_log_file = "h2_molecule_ffnn_jastrow.log"

plt.figure(figsize=(12, 6))

# 绘制 FCI 和 HF 参考线
plt.axhline(E_fci, color="red", linestyle="--", linewidth=2, label=f"FCI energy = {E_fci:.6f} Ha")
plt.axhline(E_hf, color="green", linestyle="--", linewidth=2, label=f"Hartree Fock energy = {E_hf:.6f} Ha")

# 读取并绘制 FFNN 结果
if os.path.exists(ffnn_log_file):
    with open(ffnn_log_file) as f:
        ffnn_data = json.load(f)
    x_ffnn = ffnn_data["Energy"]["iters"]
    y_ffnn = ffnn_data["Energy"]["Mean"]
    plt.plot(x_ffnn, y_ffnn, 'b-', label="FFNN only", linewidth=2)
    print(f"FFNN 最终能量: {y_ffnn[-1]:.8f} Ha")

# 读取并绘制 FFNN-Jastrow 结果
if os.path.exists(ffnn_jastrow_log_file):
    with open(ffnn_jastrow_log_file) as f:
        ffnn_jastrow_data = json.load(f)
    x_jastrow = ffnn_jastrow_data["Energy"]["iters"]
    y_jastrow = ffnn_jastrow_data["Energy"]["Mean"]
    plt.plot(x_jastrow, y_jastrow, 'm-', label="FFNN-Jastrow", linewidth=2)
    print(f"FFNN-Jastrow 最终能量: {y_jastrow[-1]:.8f} Ha")

plt.xlabel("Iterations", fontsize=12)
plt.ylabel("Energy (Ha)", fontsize=12)
plt.title("H2 Molecule: FFNN vs FFNN-Jastrow Comparison", fontsize=14)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# 打印对比总结
print("\n" + "="*60)
print("性能对比总结")
print("="*60)
print(f"FCI 能量: {E_fci:.8f} Ha")
print(f"Hartree-Fock 能量: {E_hf:.8f} Ha")
print(f"Hartree-Fock 与 FCI 误差: {abs(E_hf - E_fci):.8f} Ha")

if os.path.exists(ffnn_log_file):
    ffnn_error = abs(y_ffnn[-1] - E_fci)
    print(f"\nFFNN 最终能量: {y_ffnn[-1]:.8f} Ha")
    print(f"FFNN 与 FCI 误差: {ffnn_error:.8f} Ha")
    print(f"FFNN 相对于 HF 的改进: {abs(y_ffnn[-1] - E_hf):.8f} Ha")

if os.path.exists(ffnn_jastrow_log_file):
    jastrow_error = abs(y_jastrow[-1] - E_fci)
    print(f"\nFFNN-Jastrow 最终能量: {y_jastrow[-1]:.8f} Ha")
    print(f"FFNN-Jastrow 与 FCI 误差: {jastrow_error:.8f} Ha")
    print(f"FFNN-Jastrow 相对于 HF 的改进: {abs(y_jastrow[-1] - E_hf):.8f} Ha")
    
    if os.path.exists(ffnn_log_file):
        improvement = abs(ffnn_error - jastrow_error)
        print(f"\nFFNN-Jastrow 相对于 FFNN 的改进: {improvement:.8f} Ha")

## FFNN-Jastrow 模型说明

### 模型结构
FFNN-Jastrow 模型将波函数表示为：
```
ψ(x) = det(x) × exp(FFNN(x))
```

其中：
- **det(x)**: Slater 行列式，使用 Hartree-Fock 分子轨道系数构建
  - 提供波函数的反对称性（费米子统计）
  - 描述电子的平均场行为
  
- **FFNN(x)**: 前馈神经网络，作为 Jastrow 因子
  - 捕捉电子-电子相关性
  - 可以描述超越平均场的效应
  - 通过可学习的参数优化波函数

### 优势
1. **反对称性**: 通过行列式自动满足
2. **相关性捕捉**: FFNN 作为 Jastrow 因子可以捕捉复杂的电子相关性
3. **灵活性**: 神经网络可以学习任意形式的 Jastrow 因子
4. **可解释性**: 清晰地分离了平均场效应和关联效应

### 与纯 FFNN 的区别
- 纯 FFNN: ψ(x) = exp(FFNN(x))
  - 不满足反对称性，需要额外的约束
  - 对于费米子系统可能不够准确
  
- FFNN-Jastrow: ψ(x) = det(x) × exp(FFNN(x))
  - 自动满足反对称性
  - 更适合描述费米子系统
  - 通常能获得更低的能量